══════════════════════════════════════════════════════════════
 WEEK 8  |  DAY 7  |  AUTOMATED QA AND FILE VERIFICATION
══════════════════════════════════════════════════════════════

 LEARNING OBJECTIVES
 ───────────────────
 1. Read files programmatically and extract patterns using regex
 2. Build a check function that returns a structured list of issues
 3. Walk a directory tree and apply checks to every file

 TIME:  ~35 minutes  (3 concepts × 10-12 min each)

 YOUTUBE
 ───────
 Search: "Python os.walk directory files tutorial"
 Search: "Python regex re.search re.findall explained"
 Search: "Python ast.parse syntax checking"

 REAL-WORLD CONTEXT
 ──────────────────
 Data pipelines produce files constantly: CSVs, JSON exports, reports.
 Instead of opening each one manually, a QA script checks all of them
 automatically and tells you exactly what is wrong and where.
 This is how data teams maintain quality at scale.

══════════════════════════════════════════════════════════════


In [ ]:
import os
import re
import ast

# ─────────────────────────────────────────────────────────────
# ARCHITECTURE DECISION
# ─────────────────────
# Choosing between: pre-commit hooks vs CI/CD pipeline (GitHub Actions/Jenkins) vs manual code review only.
# Rule of thumb: use pre-commit for instant local feedback. Always add CI/CD when code is shared —
# never rely on pre-commit hooks alone for a team project.

══════════════════════════════════════════════════════════════
 CONCEPT 1 — READING A FILE AND SEARCHING FOR PATTERNS
══════════════════════════════════════════════════════════════

 open() reads a file into a string.
 re.search() looks for ONE match anywhere in the string.
 re.findall() returns a list of ALL matches.
 in operator does a simple substring check (no regex needed).

 Pattern: read → search → report

EXAMPLE ──────────────────────────────────────────────────────


In [ ]:

report_text = """
Date: 2024-01-15
Region: North
Total Sales: 48200
Status: COMPLETE
"""



Simple substring check


In [ ]:
if "COMPLETE" in report_text:
    print("Status check: PASS")
else:
    print("Status check: FAIL — missing COMPLETE marker")



Regex: find a date pattern YYYY-MM-DD


In [ ]:
date_match = re.search(r"\d{4}-\d{2}-\d{2}", report_text)
if date_match:
    print(f"Date found: {date_match.group()}")



Regex: find all numbers in the text


In [ ]:
numbers = re.findall(r"\d+", report_text)
print(f"Numbers found: {numbers}")




══════════════════════════════════════════════════════════════
 EXERCISE 1
══════════════════════════════════════════════════════════════
 You receive a pipeline log as a string.
 Write checks that print PASS or FAIL for each rule.

 Rules to check:
   - Must contain the word "SUCCESS"
   - Must contain an email address (pattern: word@word.word)
   - Must NOT contain the word "ERROR"

 Expected output:
   Status check: PASS
   Email check: PASS — found david@pipeline.com
   Error check: PASS — no errors found

--- starting data ---


In [ ]:
pipeline_log = """
Job: daily_sales_extract
Run: 2024-03-10 06:00:01
Operator: david@pipeline.com
Records loaded: 12450
Result: SUCCESS
"""






══════════════════════════════════════════════════════════════
 CONCEPT 2 — A CHECK FUNCTION THAT RETURNS A LIST OF ISSUES
══════════════════════════════════════════════════════════════

 A well-designed checker returns a LIST of issues, not just True/False.
 Empty list = file is clean.
 Non-empty list = file has problems — each item describes one problem.

 This pattern is used in pytest, data validators, linters, and
 schema checks (Great Expectations, Pydantic, etc.).

EXAMPLE ──────────────────────────────────────────────────────


In [ ]:

def check_sales_csv_row(row: dict) -> list:
    """
    Check one row from a sales CSV.
    Returns a list of issue strings. Empty list = row is valid.
    """
    issues = []

    if "amount" not in row:
        issues.append("missing 'amount' column")
    elif not isinstance(row["amount"], (int, float)):
        issues.append(f"'amount' must be a number, got: {row['amount']}")
    elif row["amount"] < 0:
        issues.append(f"'amount' is negative: {row['amount']}")

    if "region" not in row:
        issues.append("missing 'region' column")
    elif row["region"] not in ("North", "South", "East", "West"):
        issues.append(f"unknown region: {row['region']}")

    return issues




Test it


In [ ]:
rows = [
    {"amount": 1500, "region": "North"},
    {"amount": -200, "region": "East"},
    {"amount": 800,  "region": "Unknown"},
    {"region": "South"},
]

for i, row in enumerate(rows):
    issues = check_sales_csv_row(row)
    if issues:
        print(f"Row {i}: FAIL — {', '.join(issues)}")
    else:
        print(f"Row {i}: PASS")




══════════════════════════════════════════════════════════════
 EXERCISE 2
══════════════════════════════════════════════════════════════
 Write a function check_employee_record(record) that takes a dict
 and returns a list of issues based on these rules:
   - Must have keys: "name", "age", "department"
   - "age" must be a number between 18 and 70
   - "department" must be one of: "Sales", "Tech", "HR", "Finance"

 Expected output:
   Record 0: PASS
   Record 1: FAIL — age out of range: 15
   Record 2: FAIL — unknown department: Marketing
   Record 3: FAIL — missing 'age' column, unknown department: ?

--- starting data ---


In [ ]:
employee_records = [
    {"name": "Sarah Levi",   "age": 34, "department": "Tech"},
    {"name": "Omer Ben-David","age": 15, "department": "Sales"},
    {"name": "Dana Cohen",   "age": 29, "department": "Marketing"},
    {"name": "Yoav Mizrahi", "department": "?"},
]






══════════════════════════════════════════════════════════════
 CONCEPT 3 — WALKING A DIRECTORY AND CHECKING EVERY FILE
══════════════════════════════════════════════════════════════

 os.walk(folder) yields (root, subdirs, files) for every subfolder.
 Combined with a check function, you can verify an entire directory
 of files in one pass and print a structured report.

 This is the same pattern used by linters (flake8, pylint),
 test runners (pytest), and data quality platforms.

EXAMPLE ──────────────────────────────────────────────────────


In [ ]:

def check_csv_file(filepath: str) -> list:
    """Check that a CSV file is non-empty and has a header row."""
    issues = []
    try:
        with open(filepath, encoding="utf-8") as f:
            lines = f.readlines()
        if len(lines) == 0:
            issues.append("file is empty")
        elif len(lines) == 1:
            issues.append("only a header row — no data")
    except Exception as e:
        issues.append(f"could not read file: {e}")
    return issues


def run_folder_qa(folder: str):
    """Walk a folder and run check_csv_file on every .csv file."""
    total = 0
    passing = 0

    for root, dirs, files in os.walk(folder):
        dirs.sort()
        for filename in sorted(files):
            if not filename.endswith(".csv"):
                continue
            filepath = os.path.join(root, filename)
            rel = os.path.relpath(filepath, folder)
            issues = check_csv_file(filepath)
            total += 1
            if issues:
                print(f"  FAIL  {rel}")
                for issue in issues:
                    print(f"        - {issue}")
            else:
                print(f"  PASS  {rel}")
                passing += 1

    print(f"\n  {passing}/{total} files passing")




To use: run_folder_qa("path/to/your/csv/folder")
Example (runs only if a 'sample_data' folder exists here):


In [ ]:
sample_folder = os.path.join(os.path.dirname(__file__), "sample_data")
if os.path.isdir(sample_folder):
    run_folder_qa(sample_folder)
else:
    print("(run_folder_qa demo — point it at a real folder to see output)")




══════════════════════════════════════════════════════════════
 EXERCISE 3
══════════════════════════════════════════════════════════════
 Write a function run_log_qa(folder) that walks a folder,
 reads every .log file, and checks each one for these rules:
   - Must contain "SUCCESS" or "COMPLETE"
   - Must NOT contain "CRITICAL"
   - Must contain at least one date pattern YYYY-MM-DD

 Print a PASS / FAIL report like the example above.
 At the end print: "X/Y log files passing"

 You do not need real .log files to write the function.
 Write and test the logic — you can call it with any folder path.

 Expected output format:
   PASS  pipeline/daily_extract.log
   FAIL  pipeline/nightly_load.log
         - missing SUCCESS or COMPLETE marker
   FAIL  pipeline/archive/old_job.log
         - contains CRITICAL error
         - no date pattern found

   2/3 log files passing



══════════════════════════════════════════════════════════════
 CONCEPT 4 — DEVELOPER PRODUCTIVITY TOOLING: CI/CD AND LINTING
══════════════════════════════════════════════════════════════
 Production teams automate code quality — they never rely on developers
 to remember to run checks manually. Two tools handle this:

 ruff — the fastest Python linter (replaces flake8, isort, pyupgrade)
   ruff check myfile.py   → finds style errors, unused imports, bad patterns
   ruff check --fix .     → auto-fixes what it can

 pre-commit — runs checks automatically before every git commit
   .pre-commit-config.yaml defines which hooks run
   If any hook fails, the commit is rejected until the code is fixed

 GitHub Actions CI/CD — runs tests on every push to the repository
   .github/workflows/ci.yml defines the pipeline
   Every push triggers: install → lint → test → report

 Rule for this course: treat your CI pipeline as a contract.
 If it fails, the code does not ship.

 EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
import subprocess
import sys


# ── Simulate what ruff checks for ────────────────────────────
# In real use: ruff check . --select=E,F,I
# We simulate the check by inspecting code patterns

def check_code_quality(source_lines: list[str], filename: str) -> dict:
    """Simulate the checks a linter performs on a Python file."""
    issues = []

    for i, line in enumerate(source_lines, start=1):
        stripped = line.rstrip()

        if len(stripped) > 120:
            issues.append(f"  E501 line {i}: line too long ({len(stripped)} chars)")

        if "  " in stripped.lstrip() and stripped.lstrip().startswith("  "):
            pass  # indentation is fine — simplified check

        if stripped.endswith(" "):
            issues.append(f"  W291 line {i}: trailing whitespace")

        if "import *" in stripped:
            issues.append(f"  F403 line {i}: wildcard import not allowed")

        if "print(" in stripped and not stripped.strip().startswith("#"):
            pass  # allowed in exercises, flagged in production pipelines

    return {
        "file": filename,
        "issues": issues,
        "passed": len(issues) == 0,
    }


# ── Simulate a CI pipeline run ────────────────────────────────
def run_ci_pipeline(files: dict) -> bool:
    """Run quality checks on all files. Return True if all pass."""
    print("Running CI pipeline...")
    print("-" * 50)
    all_passed = True
    for filename, lines in files.items():
        result = check_code_quality(lines, filename)
        status = "PASS" if result["passed"] else "FAIL"
        print(f"  [{status}] {filename}")
        for issue in result["issues"]:
            print(issue)
        if not result["passed"]:
            all_passed = False

    print("-" * 50)
    print("CI result:", "SUCCESS" if all_passed else "FAILED — fix issues before merging")
    return all_passed


# Example: clean file passes, dirty file fails
clean_file = [
    "import pandas as pd\n",
    "\n",
    "def load_data(path: str):\n",
    "    return pd.read_csv(path)\n",
]

dirty_file = [
    "from os import *\n",           # wildcard import
    "x = 1 + 2   \n",              # trailing whitespace
    "y = " + "a" * 125 + "\n",     # line too long
]

run_ci_pipeline({"pipeline.py": clean_file, "dirty.py": dirty_file})

══════════════════════════════════════════════════════════════
 EXERCISE 4
══════════════════════════════════════════════════════════════
 Extend the check_code_quality function and run_ci_pipeline above.
 Add two more checks inside check_code_quality:

   1. Detect bare except clauses: if a line contains "except:" (no exception type)
      flag it as: "E722 line N: bare except not allowed"

   2. Detect TODO comments left in code: if a line contains "# TODO"
      flag it as: "W001 line N: TODO comment — resolve before merge"

 Then run run_ci_pipeline on the three code_samples below.
 The pipeline should FAIL for samples that have issues.

 Expected output:
     Running CI pipeline...
     --------------------------------------------------
       [PASS] clean_module.py
       [FAIL] legacy_module.py
         E722 line 3: bare except not allowed
         W001 line 6: TODO comment — resolve before merge
       [FAIL] wip_module.py
         W001 line 2: TODO comment — resolve before merge
     --------------------------------------------------
     CI result: FAILED — fix issues before merging

 --- starting data ---

In [ ]:
code_samples = {
    "clean_module.py": [
        "import pandas as pd\n",
        "\n",
        "def process(df):\n",
        "    return df.dropna()\n",
    ],
    "legacy_module.py": [
        "def risky_function(x):\n",
        "    try:\n",
        "        return 1 / x\n",
        "    except:\n",           # bare except
        "        return 0\n",
        "# TODO: add logging here\n",
    ],
    "wip_module.py": [
        "def calculate(a, b):\n",
        "    # TODO: validate inputs\n",
        "    return a + b\n",
    ],
}

# ══════════════════════════════════════════════════════════════
#  CONCEPT 5 -- CONTAINERIZATION AND CONTINUOUS INTEGRATION
# ══════════════════════════════════════════════════════════════
#
#  DOCKER -- package your app so it runs the same everywhere
#  ──────────────────────────────────────────────────────────
#  A Dockerfile describes the exact environment: OS, Python version,
#  dependencies, and start command. Anyone who runs 'docker build'
#  gets an identical environment -- no 'it works on my machine'.
#
#  Minimal Dockerfile for a FastAPI app:
#
#    FROM python:3.11-slim
#    WORKDIR /app
#    COPY requirements.txt .
#    RUN pip install -r requirements.txt
#    COPY . .
#    EXPOSE 8000
#    CMD ["uvicorn", "backend:app", "--host", "0.0.0.0", "--port", "8000"]
#
#  Key commands:
#    docker build -t myapp .        # build image from Dockerfile
#    docker run -p 8000:8000 myapp  # run container, expose port
#    docker-compose up              # start multiple services together
#
#  GITHUB ACTIONS -- run tests automatically on every push
#  ──────────────────────────────────────────────────────
#  .github/workflows/ci.yml defines a workflow that runs on push.
#  Every push to main triggers: install -> lint -> test.
#
#    name: CI
#    on: [push, pull_request]
#    jobs:
#      test:
#        runs-on: ubuntu-latest
#        steps:
#          - uses: actions/checkout@v4
#          - uses: actions/setup-python@v5
#            with: { python-version: '3.11' }
#          - run: pip install -r requirements.txt
#          - run: ruff check .
#          - run: pytest tests/
#
#  EXAMPLE ----------------------------------------------------------

In [ ]:
# ── Dockerfile and CI YAML generators ────────────────────────
# These functions produce valid config files for your projects.

def generate_dockerfile(service_name: str, port: int = 8000) -> str:
    """Return a minimal production-ready Dockerfile string."""
    return (
        f'FROM python:3.11-slim\n'
        f'WORKDIR /app\n'
        f'COPY requirements.txt .\n'
        f'RUN pip install --no-cache-dir -r requirements.txt\n'
        f'COPY . .\n'
        f'EXPOSE {port}\n'
        f'CMD ["uvicorn", "{service_name}:app", "--host", "0.0.0.0", "--port", "{port}"]\n'
    )


def generate_ci_workflow(test_command: str = 'pytest tests/') -> str:
    """Return a GitHub Actions CI workflow YAML string."""
    return (
        'name: CI\n'
        'on: [push, pull_request]\n'
        '\n'
        'jobs:\n'
        '  test:\n'
        '    runs-on: ubuntu-latest\n'
        '    steps:\n'
        '      - uses: actions/checkout@v4\n'
        '      - uses: actions/setup-python@v5\n'
        '        with:\n'
        '          python-version: "3.11"\n'
        '      - run: pip install -r requirements.txt\n'
        '      - run: ruff check .\n'
        f'      - run: {test_command}\n'
    )


print('=== Dockerfile (FastAPI backend) ===')
print(generate_dockerfile('backend'))
print('=== GitHub Actions CI workflow ===')
print(generate_ci_workflow('pytest tests/ -v'))

# ══════════════════════════════════════════════════════════════
#  EXERCISE 5
# ══════════════════════════════════════════════════════════════
#
#  Your team is deploying the NL-SQL app to a cloud server.
#  Write the infrastructure files as Python strings.
#
#  1. Write a function make_dockerfile(port: int) -> str that returns
#     a Dockerfile string for a Python app that:
#       - Uses python:3.11-slim
#       - Copies requirements.txt and installs it
#       - Copies all source files
#       - Exposes the given port
#       - Runs: uvicorn backend:app --host 0.0.0.0 --port {port}
#
#  2. Write a function make_ci_yaml(script: str) -> str that returns
#     a GitHub Actions YAML string that:
#       - Triggers on push
#       - Runs on ubuntu-latest
#       - Installs requirements.txt
#       - Runs the given script
#
#  3. Validate both outputs:
#       - Dockerfile must contain 'FROM python:3.11-slim' and str(port)
#       - CI YAML must contain 'on: [push' and the script
#
#  Expected output:
#    Dockerfile valid: True
#    CI YAML valid   : True
#    Dockerfile contains port 8080: True
#    CI step runs   : python verify_curriculum.py
#
# --- starting data ---

In [ ]:
test_port   = 8080
test_script = 'python verify_curriculum.py'